# Hotelreservering met Priority Member Middleware

Deze notebook laat zien hoe je **functie-gebaseerde middleware** gebruikt met het Microsoft Agent Framework. We bouwen voort op het voorbeeld van de conditionele workflow door een middleware-laag toe te voegen die prioriteitsleden speciale privileges geeft.

## Wat je zult leren:
1. **Functie-gebaseerde Middleware**: Resultaten van functies onderscheppen en wijzigen
2. **Toegang tot Context**: `context.result` lezen en aanpassen na uitvoering
3. **Implementatie van Businesslogica**: Voordelen voor prioriteitsleden
4. **Resultaat Overschrijven**: Functie-uitkomsten aanpassen op basis van gebruikersstatus
5. **Zelfde Workflow, Verschillende Uitkomsten**: Gedragsveranderingen gestuurd door middleware

## Workflowarchitectuur met Middleware:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## Belangrijk Verschil met Conditionele Workflow:

**Zonder Middleware** (14-conditional-workflow.ipynb):
- Parijs heeft geen kamers → Route naar alternative_agent

**Met Middleware** (deze notebook):
- Normale gebruiker + Parijs → Geen kamers → Route naar alternative_agent
- Prioriteitsgebruiker + Parijs → 🌟 Middleware overschrijft! → Beschikbaar → Route naar booking_agent

## Vereisten:
- Microsoft Agent Framework geïnstalleerd
- Kennis van conditionele workflows (zie 14-conditional-workflow.ipynb)
- GitHub-token of OpenAI API-sleutel
- Basiskennis van middlewarepatronen


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Stap 1: Definieer Pydantic-modellen voor gestructureerde outputs

Deze modellen definiëren het **schema** dat agents zullen retourneren. We hebben een veld `priority_override` toegevoegd om bij te houden wanneer middleware het beschikbaarheidsresultaat wijzigt.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Stap 2: Prioriteitsleden Database Definiëren

Voor deze demo simuleren we een prioriteitsleden database. In productie zou dit een echte database of API aanroepen.

**Prioriteitsleden:**
- `alice@example.com` - VIP-lid
- `bob@example.com` - Premium lid  
- `priority_user` - Testaccount


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## Stap 3: Maak de Hotelboekingshulpmiddel

Hetzelfde als de voorwaardelijke workflow, maar nu wordt het onderschept door middleware!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Stap 4: 🌟 Maak Priority Check Middleware (DE BELANGRIJKSTE FUNCTIE!)

Dit is de **kernfunctionaliteit** van deze notebook. De middleware:

1. **Onderbreekt** de aanroep van de hotel_booking functie
2. **Voert** de functie normaal uit door `next(context)` aan te roepen
3. **Inspecteert** het resultaat in `context.result`
4. **Overschrijft** het resultaat als de gebruiker prioriteit heeft en er geen kamers beschikbaar zijn
5. **Geeft** het gewijzigde resultaat terug aan de agent

**Belangrijk patroon:**
```python
async def my_middleware(context, next):
    await next(context)  # Functie uitvoeren
    # Nu bevat context.result de uitvoer van de functie
    if some_condition:
        context.result = new_value  # Overschrijven!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## Stap 5: Definieer Voorwaardelijke Functies voor Routing

Dezelfde voorwaardelijke functies als de voorwaardelijke workflow - ze inspecteren de gestructureerde output om routing te bepalen.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## Step 6: Maak aangepaste display executor

Zelfde executor als voorheen - toont de uiteindelijke workflow-uitvoer.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## Stap 7: Laad omgevingsvariabelen

Configureer de LLM-client (GitHub Models of OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Stap 8: Maak AI-agents met Middleware

**BELANGRIJK VERSCHIL:** Bij het aanmaken van de availability_agent geven we de `middleware`-parameter mee!

Dit is hoe we de priority_check_middleware in de functie-aanroep-pijplijn van de agent injecteren.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Stap 9: Bouw de Workflow

Zelfde workflow-structuur als voorheen - conditionele routering op basis van beschikbaarheid.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## Stap 10: Testgeval 1 - Gewone gebruiker in Parijs (Geen Override)

Een gewone gebruiker probeert Parijs te boeken → Geen kamers → Route naar alternative_agent


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Stap 11: Testgeval 2 - 🌟 Prioriteitsgebruiker in Parijs (MET Override!)

Een prioriteitslid probeert Parijs te boeken → In eerste instantie geen kamers → 🌟 Middleware overschrijft! → Stuurt naar booking_agent

**Dit is de belangrijkste demonstratie van middlewarekracht!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## Stap 12: Testgeval 3 - Prioriteitsgebruikers in Stockholm (Reeds beschikbaar)

Prioriteitsgebruiker probeert Stockholm → Kamers beschikbaar → Geen override nodig → Verwijst naar booking_agent

Dit toont aan dat de middleware alleen in actie komt wanneer het nodig is!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## Belangrijke inzichten en middleware concepten

### ✅ Wat je hebt geleerd:

#### **1. Functie-gebaseerd middleware patroon**

Middleware onderschept functieverzoeken met een simpele async functie:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # Voor de uitvoering van de functie
    print("Intercepting...")
    
    # Voer de functie uit
    await next(context)
    
    # Na de uitvoering van de functie - inspecteer resultaat
    if context.result:
        # Pas resultaat aan indien nodig
        context.result = modified_value
```

#### **2. Toegang tot context en resultaat overschrijven**

- `context.function` - Toegang tot de aan te roepen functie
- `context.arguments` - Lees de functieargumenten
- `context.kwargs` - Toegang tot extra parameters
- `await next(context)` - Voer de functie uit
- `context.result` - Lees/wijzig de uitvoer van de functie

#### **3. Implementatie van bedrijfslogica**

Onze middleware implementeert prioriteitsvoordelen voor leden:
- **Reguliere gebruikers**: Geen aanpassingen, standaard workflow
- **Prioriteitsgebruikers**: Overschrijf "geen beschikbaarheid" → "beschikbaar"
- **Voorwaardelijke logica**: Overschrijft alleen wanneer nodig

#### **4. Zelfde workflow, verschillende uitkomsten**

De kracht van middleware:
- ✅ Geen wijzigingen in de workflowstructuur
- ✅ Geen wijzigingen in de toolfunctie
- ✅ Geen wijzigingen in voorwaardelijke routeringslogica
- ✅ Alleen middleware → Verschillend gedrag!

### 🚀 Toepassingen in de praktijk:

1. **VIP/Premium functies**
   - Overschrijf limieten voor premium gebruikers
   - Voorzie prioriteits toegang tot resources
   - Ontgrendel premium functies dynamisch

2. **A/B testen**
   - Leid gebruikers naar verschillende implementaties
   - Test nieuwe functies met specifieke gebruikers
   - Geleidelijke uitrol van functies

3. **Beveiliging & naleving**
   - Controleer functieaanroepen
   - Blokkeer gevoelige handelingen
   - Handhaaf bedrijfsregels

4. **Prestatieoptimalisatie**
   - Cache resultaten voor specifieke gebruikers
   - Sla dure operaties over waar mogelijk
   - Dynamische toewijzing van resources

5. **Foutafhandeling & herhaling**
   - Vang en behandel fouten soepel af
   - Implementeer retry-logica
   - Val terug op alternatieve implementaties

6. **Logging & monitoring**
   - Volg executietijden van functies
   - Log parameters en resultaten
   - Monitor gebruikspatronen

### 🔑 Belangrijkste verschillen met decorators:

| Kenmerk | Decorator | Middleware |
|---------|-----------|------------|
| **Scope** | Één enkele functie | Alle functies in agent |
| **Flexibiliteit** | Vast bij definitie | Dynamisch tijdens runtime |
| **Context** | Beperkt | Volledige agent context |
| **Samenstelling** | Meerdere decorators | Middleware pijplijn |
| **Agent-aware** | Nee | Ja (toegang tot agentstatus) |

### 📚 Wanneer middleware gebruiken:

✅ **Middleware gebruiken wanneer:**
- Je gedrag wilt aanpassen op basis van gebruiker-/sessiestatus
- Je logica op meerdere functies wilt toepassen
- Je toegang tot context op agentniveau nodig hebt
- Je cross-cutting concerns implementeert (logging, auth, etc.)

❌ **Middleware niet gebruiken wanneer:**
- Simpele inputvalidatie (gebruik Pydantic)
- Functie-specifieke logica (houd binnen functie)
- Eenmalige aanpassingen (verander gewoon de functie)

### 🎓 Geavanceerde patronen:

```python
# Meerdere middleware (uitvoeringsvolgorde is belangrijk!)
middleware=[
    logging_middleware,      # Logt eerst
    auth_middleware,         # Controleert dan authenticatie
    cache_middleware,        # Controleert dan cache
    rate_limit_middleware,   # Past dan snelheidsbeperking toe
    priority_check_middleware  # Ten slotte prioriteitscontrole
]

# Voorwaardelijke middleware-uitvoering
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # Resultaat aanpassen
    else:
        # Voer uitvoering volledig over.
        context.result = cached_value
```

### 🔗 Gerelateerde concepten:

- **Agent Middleware**: Onderschept agent.run() aanroepen
- **Functie Middleware**: Onderschept tool functie-aanroepen (wat wij gebruikten!)
- **Middleware pijplijn**: Ketting van middleware die in volgorde worden uitgevoerd
- **Context propagatie**: Staat doorgeven door middleware keten


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
